<a href="https://colab.research.google.com/github/rorisDS/workshop_ai_agents/blob/develop/notebooks_es/AgentesEnAcci%C3%B3n_ElPoderDeLasToolsYLaOrquestaci%C3%B3n.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Agentes en Acción: El poder de las Tools y la Orquestación.

Anteriormente hemos visto cómo los Agentes de IA pueden razonar para resolver tareas lógicas en el dominio digital. Sin embargo, el verdadero poder de la "Agencia" surge cuando el modelo deja de ser un sistema aislado y empieza a utilizar las Tools como una extensión de sus propias capacidades.

En este ejercicio, daremos un salto cualitativo. Vamos a explorar cómo el uso estratégico de herramientas nos permite dotar a un Agente de facultades que transforman por completo su utilidad:

* Percepción: La capacidad de observar y medir el mundo físico.

* Intervención: La capacidad de actuar sobre el entorno y generar cambios tangibles.

* Delegación: La capacidad de recurrir a expertos para resolver tareas cognitivas.

A través de varios casos de uso independientes, descubriremos cómo estas piezas —que inicialmente parecen simples funciones— pueden evolucionar hasta convertirse en los pilares de sistemas autónomos mucho más complejos y colaborativos.

## Instalación de librerías

In [ ]:
!pip install langchain==1.2.7
!pip install langchain-core==1.2.7
!pip install langchain-openai==1.1.7  # Para usar modelos de OpenAI
!pip install langchain-google-genai==4.2.0  # Para usar modelos de Google (Gemini)
!pip install ddgs==9.10.0
!pip install langchain-community==0.4.1

# Limpia output
from IPython.display import clear_output
clear_output()

## LLM

In [ ]:
# Use Google Colab Secrets
import os
try:
    from google.colab import userdata
    os.environ["OPENAI_API_KEY"] = userdata.get('OPENAI_API_KEY')
    os.environ["GOOGLE_API_KEY"] = userdata.get('GOOGLE_API_KEY')
except:
    pass

* Conectando a un modelo de OpenAI

   - Crear API Key: https://platform.openai.com/docs/quickstart
   - Seleccionar un modelo: https://platform.openai.com/docs/models

In [ ]:
import getpass
import os

if not os.environ.get("OPENAI_API_KEY"):
    os.environ["OPENAI_API_KEY"] = getpass.getpass("Enter your OpenAI API key: ")

from langchain_openai import ChatOpenAI
llm = ChatOpenAI(
    model="gpt-4o",  # Seleccionamos un modelo por su keyword
    temperature=0,
    max_tokens=None,
    # other params...
    )

* Conectando a un modelo de Google
   - Crear API Key: https://ai.google.dev/gemini-api/docs/api-key
   - Seleccionar un modelo: https://ai.google.dev/gemini-api/docs/models

In [ ]:
import getpass
import os

if "GOOGLE_API_KEY" not in os.environ:
    os.environ["GOOGLE_API_KEY"] = getpass.getpass("Enter your Google AI API key: ")

from langchain_google_genai import ChatGoogleGenerativeAI

llm = ChatGoogleGenerativeAI(
    model="gemini-3-flash-preview",
    temperature=0.0,
    max_tokens=None,
    # other params...
)

* Obtener la respuesta final



Cada proveedor de LLM (OpenAI, Google, etc.) estructura sus respuestas de forma ligeramente distinta dentro de LangChain. Mientras que algunos devuelven un string directo, otros (como Gemini) utilizan un formato de lista multimodal para soportar texto, imágenes o audio.

Creamos esta función auxiliar para "normalizar" la salida, extrayendo únicamente el contenido textual independientemente del modelo que estemos utilizando. Así, garantizamos que nuestro sistema sea agnóstico al proveedor.

In [ ]:
def get_clean_response(result):
    # Obtenemos el último mensaje de la lista
    last_message = result["messages"][-1]
    content = last_message.content

    # Si es una lista (caso Gemini), extraemos el texto del primer elemento
    if isinstance(content, list):
        # Buscamos el primer elemento que tenga la llave 'text'
        for item in content:
            if isinstance(item, dict) and 'text' in item:
                return item['text']
            elif isinstance(item, str):
                return item

    # Si ya es un string (caso OpenAI), lo devolvemos tal cual
    return content

## Agente de Lectura: Especialista en Diagnóstico Físico

Un Agente de IA es un sistema capaz de observar su entorno a través de herramientas. Estas herramientas permiten al modelo percibir el mundo físico por medio de medidas de sensores que, de otro modo, serían inaccesibles para el LLM.

En este apartado, hemos construido un agente que asume el rol de un analista técnico. Su única misión es acceder a fuentes de datos heterogéneas (dispositivos inteligentes) y transformar datos crudos en un informe de diagnóstico estructurado.

Para que el agente pueda "sentir" el estado del usuario, hemos simulado dos dispositivos independientes que conectan el modelo con información real del entorno:

* **Báscula Inteligente**: Proporciona telemetría sobre el peso y la composición corporal.

* **Smartwatch**: Proporciona métricas biométricas como el sueño, la frecuencia cardíaca, el nivel de actividad y el estrés.

El objetivo de este componente es demostrar cómo el razonamiento del modelo orquesta la llamada a estos sensores para construir un contexto sólido antes de emitir cualquier juicio.


In [ ]:
# Datos provenientes de la API de la báscula inteligente (ej. Withings)
SMART_SCALE_DATA = {
    "weight_log": [78.5, 78.2, 77.9, 77.5, 77.0], # Histórico en kg
    "body_fat_percentage": 22.5
}

# Datos provenientes de la API del Smartwatch (ej. Apple Health / Garmin)
SMARTWATCH_DATA = {
    "sleep_hours": [7.5, 6.8, 8.2, 5.5, 7.0],
    "resting_heart_rate": 64,
    "daily_steps": [10500, 12000, 9800, 4500, 11000],
    "stress_level": "Medio" # Basado en HRV
}

In [ ]:
from langchain_core.tools import tool

@tool
def get_weight_trend():
    """
    Consulta el histórico de peso en la báscula inteligente.
    Útil para detectar tendencias de pérdida o ganancia de masa corporal.
    """
    return f"Últimos registros de peso (kg): {SMART_SCALE_DATA['weight_log']}. Porcentaje de grasa actual: {SMART_SCALE_DATA['body_fat_percentage']}%"

@tool
def get_sleep_analysis():
    """
    Recupera las horas de sueño del smartwatch.
    Esencial para evaluar la recuperación biológica y el estado de fatiga del usuario.
    """
    return f"Horas de sueño de los últimos días: {SMARTWATCH_DATA['sleep_hours']}"

@tool
def get_cardiovascular_data():
    """
    Accede a las pulsaciones en reposo y nivel de estrés del smartwatch.
    Útil para identificar sobreentrenamiento o falta de descanso severa.
    """
    return (f"Frecuencia cardíaca en reposo: {SMARTWATCH_DATA['resting_heart_rate']} ppm. "
            f"Nivel de estrés detectado: {SMARTWATCH_DATA['stress_level']}")

@tool
def get_movement_activity():
    """
    Obtiene el conteo de pasos diarios.
    Sirve para cuantificar el nivel de actividad física real frente al sedentarismo.
    """
    return f"Pasos diarios de la última semana: {SMARTWATCH_DATA['daily_steps']}"

In [ ]:
from langchain.agents import create_agent

system_prompt = """Eres un Especialista en Diagnóstico Físico. Tu única misión es recopilar datos de los sensores disponibles y generar un informe objetivo y detallado sobre el estado actual del usuario.

Tu metodología:

Observa las tendencias: ¿El peso sube o baja? ¿El sueño mejora o empeora?

Cruza datos: Relaciona la falta de sueño con los niveles de actividad o frecuencia cardíaca.

Concluye con un 'Estado de Salud Actual' claro (ej: 'En recuperación', 'Sobreentrenado', 'Estable').

No des consejos médicos, limítate a analizar los datos existentes."""

physical_agent = create_agent(
    model=llm,
    tools=[get_weight_trend, get_sleep_analysis, get_cardiovascular_data, get_movement_activity],
    system_prompt=system_prompt
)

In [ ]:
from langchain.messages import HumanMessage

result = physical_agent.invoke(
    {
        "messages": [
            HumanMessage("Realiza un análisis de mi estado físico actual basándote en mis dispositivos.")
        ]
    }
)

In [ ]:
# Print the conversation
for message in result["messages"]:
    if hasattr(message, 'pretty_print'):
        message.pretty_print()
    else:
        print(f"{message.type}: {message.content}")

================================ Human Message =================================

Realiza un análisis de mi estado físico actual basándote en mis dispositivos.
================================== Ai Message ==================================
Tool Calls:
  get_weight_trend (call_zAt5ftzn6uXNpoAzJUyCnclY)
 Call ID: call_zAt5ftzn6uXNpoAzJUyCnclY
  Args:
  get_sleep_analysis (call_U2z9lk6qBbRXItqg18CHLYj2)
 Call ID: call_U2z9lk6qBbRXItqg18CHLYj2
  Args:
  get_cardiovascular_data (call_sP8qJRqwYxTjJtY0JLyIYLza)
 Call ID: call_sP8qJRqwYxTjJtY0JLyIYLza
  Args:
  get_movement_activity (call_TsfwP0h7jYHglY7EXxUICwHZ)
 Call ID: call_TsfwP0h7jYHglY7EXxUICwHZ
  Args:
================================= Tool Message =================================
Name: get_weight_trend

Últimos registros de peso (kg): [78.5, 78.2, 77.9, 77.5, 77.0]. Porcentaje de grasa actual: 22.5%
================================= Tool Message =================================
Name: get_sleep_analysis

Horas de sueño de los últim

In [ ]:
print(f"Respuesta: {get_clean_response(result)}")

Respuesta: **Análisis de Estado Físico Actual**

1. **Tendencia de Peso:**
   - El peso ha mostrado una tendencia a la baja: de 78.5 kg a 77.0 kg en los registros más recientes.
   - El porcentaje de grasa corporal actual es del 22.5%.

2. **Análisis del Sueño:**
   - Las horas de sueño han sido variables: 7.5, 6.8, 8.2, 5.5 y 7.0 horas.
   - Se observa una noche con sueño significativamente bajo (5.5 horas), lo que podría indicar un descanso insuficiente en ese día.

3. **Datos Cardiovasculares:**
   - La frecuencia cardíaca en reposo es de 64 ppm, lo cual es un valor normal.
   - El nivel de estrés detectado es medio, lo que podría estar relacionado con la variabilidad en las horas de sueño.

4. **Actividad Física:**
   - El conteo de pasos diarios muestra una actividad física moderada, con un día de baja actividad (4500 pasos).

**Estado de Salud Actual: Estable**

El usuario presenta una tendencia de pérdida de peso, con un nivel de actividad física moderado. Las horas de sueño son

## Agente de Acción: Agente de compra

Si el agente anterior nos servía para demostrar la percepción, el Agente de Acción tiene como objetivo demostrar que un LLM puede intervenir y modificar su entorno. Un Agente no solo razona sobre la información, sino que puede ejecutar cambios que tienen un impacto físico, como realizar un pedido que llegará al domicilio del usuario.

En este apartado, el agente asume una responsabilidad operativa y logística. No se limita a sugerir productos, sino que debe cerrar el ciclo de ejecución gestionando dos variables críticas del mundo real:

* Inventario Externo: El agente debe explorar un catálogo de productos disponible en una tienda online para encontrar aquellos que cumplan con los requisitos nutricionales.

* Restricciones Financieras (Cartera): La acción está sujeta a un presupuesto limitado. El agente debe consultar su saldo, priorizar compras y actualizar el estado de su cartera tras la ejecución.

El éxito de este componente se mide por su capacidad de pasar de la intención (querer comprar) a la transacción (ejecutar la compra), demostrando que la IA puede ser un actor funcional en procesos de negocio y logística.

In [ ]:
# Simulación de la base de datos de una tienda online
ONLINE_STORE_INVENTORY = [
    # PROTEÍNAS
    {"id": "pro_001", "name": "Pack Pechugas de Pollo", "tags": ["proteína", "bajo en grasa"], "price": 5.50},
    {"id": "pro_003", "name": "Salmón Noruego Fresco", "tags": ["proteína", "omega-3", "saludable"], "price": 18.20}, # Opción cara
    {"id": "pro_004", "name": "Tofu Orgánico", "tags": ["proteína vegetal", "vegano"], "price": 4.10},
    {"id": "pro_005", "name": "Entrecot de Ternera", "tags": ["proteína", "hierro", "alto en grasa"], "price": 14.50},

    # VEGETALES Y FRUTAS
    {"id": "veg_001", "name": "Bolsa Espinacas Baby", "tags": ["fibra", "vitaminas"], "price": 2.00},
    {"id": "veg_002", "name": "Ensalada César preparada", "tags": ["vegetales", "salsa", "alto en sodio"], "price": 4.50},
    {"id": "veg_003", "name": "Brócoli al vapor", "tags": ["fibra", "vitaminas", "bajo en calorías"], "price": 2.80},
    {"id": "fru_001", "name": "Cesta de Plátanos (5 ud)", "tags": ["potasio", "energía"], "price": 3.00},
    {"id": "fru_002", "name": "Arándanos frescos", "tags": ["antioxidantes", "vitaminas"], "price": 6.20},

    # CARBOHIDRATOS Y "CHEAT MEALS"
    {"id": "car_001", "name": "Arroz Integral 1kg", "tags": ["carbohidratos complejos", "fibra"], "price": 2.30},
    {"id": "pro_002", "name": "Pizza Barbacoa Familiar", "tags": ["carbohidratos", "alto en sodio", "procesado"], "price": 12.00},
    {"id": "junk_001", "name": "Pack de Donuts (4 ud)", "tags": ["azúcar", "ultraprocesado"], "price": 3.50},
    {"id": "junk_002", "name": "Bebida Energética", "tags": ["cafeína", "azúcar"], "price": 1.50},

    # COMPLEMENTOS
    {"id": "sup_001", "name": "Multivitamínico 30 cápsulas", "tags": ["suplemento", "vitaminas"], "price": 15.00},
    {"id": "bev_001", "name": "Agua Mineral 1.5L", "tags": ["hidratación"], "price": 0.80}
]

# Estado del mundo: Presupuesto inicial
WALLET_BALANCE = 20.0  # Euros

In [ ]:
from langchain_core.tools import tool

@tool
def get_wallet_balance():
    """
    Consulta el dinero disponible en la cartera del usuario.
    Es obligatorio consultar esto antes de intentar realizar cualquier compra.
    """
    return f"Saldo actual: {WALLET_BALANCE}€"

@tool
def get_available_products():
    """
    Devuelve la lista completa de productos disponibles en la tienda con sus nombres y descripciones.
    Útil para que el agente vea qué opciones tiene antes de decidir.
    """
    # Solo devolvemos lo necesario para que el LLM decida
    return [{"id": p["id"], "name": p["name"], "tags": p["tags"]} for p in ONLINE_STORE_INVENTORY]

@tool
def place_order(product_id: str):
    """
    Realiza la compra del producto. Esta herramienta verifica si hay saldo suficiente,
    realiza el pedido y descuenta el dinero de la cartera.
    """
    global WALLET_BALANCE

    # Buscamos el producto en el inventario anterior
    product = next((p for p in ONLINE_STORE_INVENTORY if p["id"] == product_id), None)

    if not product:
        return "Error: Producto no encontrado."

    if WALLET_BALANCE >= product["price"]:
        WALLET_BALANCE -= product["price"]
        return (f"COMPRA EXITOSA: {product['name']}. "
                f"Precio: {product['price']}€. "
                f"Saldo restante: {WALLET_BALANCE}€.")
    else:
        return f"ERROR: Saldo insuficiente. El producto cuesta {product['price']}€ y solo tienes {WALLET_BALANCE}€."

In [ ]:
from langchain.agents import create_agent

system_prompt = """Eres el Gestor de Suministros Nutricionales del usuario.
Tu responsabilidad principal es asegurar que el usuario reciba en su domicilio los alimentos que mejor se adapten a las recomendaciones dietéticas que te sean entregadas.

Para cumplir con tu misión, debes gestionar de forma autónoma el presupuesto disponible en la cartera, asegurándote siempre de que ninguna recomendación se quede sin ejecutar por falta de iniciativa.
Tienes autoridad total para seleccionar productos sustitutivos si el saldo es insuficiente o si un producto específico no está disponible, siempre que respetes la lógica nutricional.
Tu objetivo final no es solo asesorar, sino garantizar que la compra se efectúe con éxito mediante las herramientas de pedido."""

shopping_agent = create_agent(
    model=llm,
    tools=[get_wallet_balance, get_available_products, place_order],
    system_prompt=system_prompt
)

In [ ]:
from langchain.messages import HumanMessage

result = shopping_agent.invoke(
    {
        "messages": [
            HumanMessage("El usuario necesita recuperar potasio y evitar grasas saturadas.")
        ]
    }
)

In [ ]:
from langchain.messages import HumanMessage

result = shopping_agent.invoke({"input": """dietary_plan: 1. Proteínas Magras: pollo, pavo, pescado, legumbres.
2. Grasas Saludables: aguacates, nueces, semillas, aceite de oliva.
3. Carbohidratos de Absorción Lenta: avena, quinoa, batatas, arroz integral.
4. Alimentos Ricos en Magnesio: espinacas, almendras, plátanos.
5. Alimentos con Triptófano: pavo, huevos, semillas de calabaza.
6. Hidratación: agua.
7. Alimentos Ricos en Antioxidantes: bayas, verduras de hojas verdes.
8. Infusiones Relajantes: manzanilla, valeriana."""
    }
)

In [ ]:
# Print the conversation
for message in result["messages"]:
    if hasattr(message, 'pretty_print'):
        message.pretty_print()
    else:
        print(f"{message.type}: {message.content}")

================================== Ai Message ==================================

Por favor, proporciona las recomendaciones dietéticas que necesitas que gestione para ti.


In [ ]:
print(f"Respuesta: {get_clean_response(result)}")

Respuesta: He realizado con éxito las siguientes compras para tu plan dietético:

- **Pack Pechugas de Pollo**: 5.5€
- **Arroz Integral 1kg**: 2.3€
- **Bolsa Espinacas Baby**: 2.0€
- **Cesta de Plátanos (5 ud)**: 3.0€
- **Agua Mineral 1.5L**: 0.8€

Lamentablemente, no fue posible comprar los **Arándanos frescos** debido a saldo insuficiente. Si deseas ajustar el pedido o buscar alternativas, por favor házmelo saber. Tu saldo restante es de 0.60€.


## Tool Cognitiva: Experto Dietista (LLM-as-a-Tool)

Hasta ahora, hemos utilizado herramientas para conectar el modelo con el entorno físico. Sin embargo, las tools también pueden ser utilizadas para delegar **tareas cognitivas** especializadas a otros modelos de lenguaje (o al mismo modelo bajo un rol distinto).

En este apartado, encapsulamos un **Experto Dietista** dentro de una herramienta. El objetivo es demostrar cómo podemos aislar una capacidad intelectual específica —en este caso, el análisis nutricional— para que pueda ser invocada de forma independiente, tal como haríamos con una base de datos o un sensor.

In [ ]:
from langchain_core.messages import SystemMessage, HumanMessage

@tool
def nutritionist_expert_review(health_report: str):
    """
    Recibe un informe de salud detallado y devuelve una estrategia nutricional.
    Es un experto en convertir datos médicos en recomendaciones de alimentos específicos.
    """

    system_prompt = (
        "Eres un Nutricionista Clínico de élite. Tu trabajo es recibir informes "
        "de salud y traducirlos en una lista de necesidades "
        "alimenticias concretas. No menciones marcas, menciona tipos de alimentos "
        "(ej: 'fruta con potasio', 'proteína magra', 'hidratos de absorción lenta')."
    )

    # Invocamos directamente al LLM (sin bucles de agentes, una sola pasada)
    messages = [
        SystemMessage(content=system_prompt),
        HumanMessage(content=f"Aquí tienes el informe de salud para analizar: {health_report}")
    ]

    response = llm.invoke(messages)
    return get_clean_response(response)

In [ ]:
# Prueba manual para el workshop
informe_ejemplo = "El usuario ha dormido 4 horas y tiene las pulsaciones altas."
recomendacion = nutritionist_expert_review.invoke(informe_ejemplo)

print(f"SALIDA DEL DIETISTA:\n{recomendacion}")

SALIDA DEL DIETISTA:
Para abordar la falta de sueño y las pulsaciones altas, es importante centrarse en alimentos que promuevan la relajación, el equilibrio energético y la salud cardiovascular. Aquí tienes una lista de necesidades alimenticias concretas:

1. **Carbohidratos de absorción lenta**: Opta por avena, quinoa o batatas para proporcionar energía sostenida y evitar picos de azúcar en sangre que puedan aumentar las pulsaciones.

2. **Proteínas magras**: Incluye pollo, pavo o pescado como el salmón, que también aporta ácidos grasos omega-3 beneficiosos para el corazón.

3. **Frutas ricas en potasio**: Como plátanos o aguacates, que ayudan a regular la presión arterial y pueden contribuir a reducir las pulsaciones.

4. **Verduras de hoja verde**: Espinacas o kale, que son ricas en magnesio, un mineral que ayuda a relajar los músculos y puede mejorar la calidad del sueño.

5. **Alimentos ricos en triptófano**: Como nueces, semillas de calabaza o pavo, que pueden ayudar a mejorar el

## El Gestor de Estilo de Vida: Patrón Multi-Agente

Un **sistema Multi-Agente** es aquel en el que múltiples agentes de IA interaccionan y colaboran para alcanzar un objetivo complejo. Siguiendo un **patrón de sub-agentes**, haremos que agentes especializados lleven a cabo tareas específicas mientras un agente "padre" los coordina.

<img src="https://raw.githubusercontent.com/rorisDS/workshop_ai_agents/refs/heads/develop/images/pattern-subagents.png" width=600>

En este ejemplo, convertimos a nuestros especialistas previos (el Analista Físico y el Agente de Compra) y a nuestra unidad cognitiva (el Dietista) en herramientas (tools) de un Agente Superior. Este "Agente Padre" u Orquestador no necesita saber cómo leer un sensor o cómo gestionar un carrito de la compra; su función es la **estrategia y la delegación**.

Ventajas del patrón de Sub-Agentes:
* **Abstracción**: El orquestador maneja objetivos de alto nivel, mientras que la complejidad técnica queda encapsulada dentro de cada herramienta.

* **Modularidad**: Podemos actualizar, cambiar o añadir nuevos especialistas (ej. un agente de entrenamiento o de agenda) sin necesidad de reescribir la lógica central.

* **Especialización**: Cada sub-agente mantiene su propio contexto y reglas, evitando que el exceso de información confunda al modelo principal.

Al invocar al **Gestor de Estilo de Vida**, el modelo utiliza su "razonamiento" para decidir la secuencia lógica: percibir el estado físico, consultar al experto y ejecutar la acción en el mundo real. Se convierte en un sistema autónomo completo.

In [ ]:
from langchain_core.tools import tool

@tool
def analyze_user_physical_state(query: str):
    """
    Consulta al Especialista Físico para obtener un informe basado en sensores
    (peso, sueño, actividad). Útil para entender el estado actual del usuario.
    """
    # Invocamos al agente que ya definimos antes
    result = physical_agent.invoke(
        {
            "messages": [
                HumanMessage(content=query)
            ]
        }
    )
    return get_clean_response(result)

@tool
def execute_food_shopping(dietary_plan: str):
    """
    Activa al Agente de Compras para buscar productos y realizar el pedido real
    basándose en el plan dietético. Gestiona la cartera y el stock.
    """
    # Invocamos al agente de acción
    result = shopping_agent.invoke(
        {
            "messages": [
                HumanMessage(content=dietary_plan)
            ]
        }
    )
    return get_clean_response(result)


In [ ]:
# --- EL ORQUESTADOR ---

# Lista de herramientas usando la misma lógica que los sub-agentes
orchestrator_tools = [
    analyze_user_physical_state,
    nutritionist_expert_review,
    execute_food_shopping
]

life_manager_agent = create_agent(
    model=llm,
    tools=orchestrator_tools,
    system_prompt=(
        "Eres un Asistente Personal Autónomo. Tu misión es resolver las necesidades del usuario "
        "coordinando a tus expertos internos. No solo informes, asegúrate de que las acciones "
        "se ejecuten hasta el final (compras, pedidos, etc.) cuando sea necesario."
    )
)

In [ ]:
from langchain.messages import HumanMessage

result = life_manager_agent.invoke(
    {
        "messages": [
            HumanMessage("Haz la compra de alimentos en base a mis necesidades físicas")
        ]
    }
)

In [ ]:
# Print the conversation
for message in result["messages"]:
    if hasattr(message, 'pretty_print'):
        message.pretty_print()
    else:
        print(f"{message.type}: {message.content}")

================================ Human Message =================================

Haz la compra de alimentos en base a mis necesidades físicas
================================== Ai Message ==================================
Tool Calls:
  analyze_user_physical_state (call_rlPeYIaxo5Sn1G78PHkNadtB)
 Call ID: call_rlPeYIaxo5Sn1G78PHkNadtB
  Args:
    query: Necesidades físicas actuales del usuario para planificar la compra de alimentos
================================= Tool Message =================================
Name: analyze_user_physical_state

No puedo proporcionar recomendaciones sobre necesidades físicas o dietéticas. Sin embargo, puedo analizar los datos de salud disponibles para ofrecer un informe sobre el estado actual del usuario. Esto puede incluir tendencias de peso, patrones de sueño, datos cardiovasculares y niveles de actividad física. ¿Te gustaría que proceda con este análisis?
================================== Ai Message ==================================

Parece que h

In [ ]:
print(f"Respuesta: {get_clean_response(result)}")

Respuesta: Actualmente, no hay suficiente saldo disponible para realizar la compra de los alimentos recomendados. Te sugiero recargar tu cartera para poder adquirir los productos necesarios según el plan dietético. Si necesitas ayuda con otra cosa o deseas ajustar el plan de compras, házmelo saber.


## Conclusiones: La Anatomía de la Agencia

En este workshop hemos desmitificado la construcción de Agentes de IA, pasando de un modelo que solo "responde preguntas" a un sistema que "interactúa con el mundo". A través de los ejemplos, hemos validado tres pilares fundamentales:

1. **Interacción con el Entorno (Lectura y Acción)**: Los tools son los sentidos y las extremidades del modelo. Hemos demostrado que un agente puede percibir la realidad a través de sensores (lectura) y modificarla a través de actuadores (acciones físicas como una compra).

2. **Especialización Cognitiva**: Hemos visto que un LLM puede ser encapsulado en una tool para realizar labores intelectuales aisladas. Esto nos permite delegar tareas de análisis o revisión a "expertos" sin contaminar el flujo principal de razonamiento.

3. **Escalabilidad Multi-Agente**: En el paso final, hemos probado que una tool puede ser, en sí misma, otro agente completo. Este patrón de sub-agentes es el que permite construir sistemas de alta complejidad, donde la orquestación y la delegación sustituyen a la programación lineal.

**En definitiva, hemos pasado de programar secuencias de pasos a diseñar ecosistemas de capacidades**. La verdadera potencia de los agentes no reside en el modelo que elijamos, sino en cómo diseñamos las tool y la colaboración entre ellas.